# Lab 5 · Counting your way to a p-value

**Today:** You will be able to compute a p-value with a loop and a counter.

**Before you start:** Read Chapter 2.

Each section names one idea, explains what it does, and asks you to **predict
what a cell prints before you run it**. Write the prediction down, on paper, out
loud, or in a comment. A prediction you can compare against the output is what
tells you which parts of the code you can already read.

Most sections end with a **Test your understanding** task: write a small piece
of code, then run the check cell under it. Every task has a hint in the **Hints**
block at the end of the notebook, for when you want it. The check never grades and never
breaks anything. A ⬜ means not attempted yet, a ❌ means not yet and comes with
a hint, and a ✅ means passing. Run the check cells rather than editing them.
Everything else in the notebook is yours to change.

**AI in this lab.** Until your prediction is written down, work at level 1, with
no AI. The prediction is how you find out what you can read unaided, and both
exams are level 1. Once you have run a cell, level 3 is encouraged: ask your
tutor to explain anything you missed.

Run every cell, and change things to see what happens. Nothing in this notebook
can be broken in a way that matters.

## 1 · An observation, and a question

You flipped a coin 30 times and got 22 heads. Chapter 2 asks the question: if the coin were fair, how often would 30 flips look at least this extreme?

First, simulate the fair coin, which represents the scenario where we know the truth by construction. Compare this function to lab 3's count_heads.

In [ ]:
import numpy as np

def count_heads(n_flips, chance, rng):
    heads = 0
    for _ in range(n_flips):
        if rng.random() < chance:
            heads += 1
    return heads

rng = np.random.default_rng(37)
print("five fair-coin runs of 30:",
      count_heads(30, 0.5, rng), count_heads(30, 0.5, rng),
      count_heads(30, 0.5, rng), count_heads(30, 0.5, rng),
      count_heads(30, 0.5, rng))

Simulating five fair coins gave 30 flips each, and none reached 22 heads or more. But five simulations aren't enough to say how often that happens. For a more reliable estimate, we need thousands of simulated runs, and a count of how many of them reach at least 22 heads.

**Test your understanding.** Create a variable named `n_extreme` that stores one whole number: how many of 5,000 simulated fair-coin runs of 30 flips came out **at 22 heads or more**. Use the generator the cell below creates with seed 41, and create it only once. The check reads `n_extreme` as a stored number and does not call it, so do not write it as a function. This is question 1. Its hint is at the end of the notebook.

In [ ]:
# your turn: a variable named n_extreme, one whole number
# of 5000 fair runs (seed 41), how many reached 22+ heads?
import numpy as np
rng = np.random.default_rng(41)

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("n_extreme", expect=50,
      hint="one count_heads(30, 0.5, rng) per run, all 5000 from the seed-41 generator; count runs where the result >= 22")

## 2 · The fraction is the answer

Your counter divided by the number of runs is the fraction of fair-coin simulations at least as extreme as what you saw. Chapter 2 adds one small correction to prevent a p-value of exactly 0:

    p = (n_extreme + 1) / (n_runs + 1)

That is a **p-value**. Not the probability the coin is fair but the frequency with which fairness produces data like yours. Run the arithmetic on your `n_extreme` from section 1 — predict its rough size first (50-ish out of 5,000 is about what percent?).

In [ ]:
print("p =", (50 + 1) / (5000 + 1))

About 0.0102 — roughly one fair-coin simulation in a hundred looks like yours and it was calculated with a loop, an if, a counter, and division. The interpretation depends on the context, but this basic computation underlies the concept of a p-value.

**Now in one call.** The last lab drew whole experiments in a single line, and
`rng.binomial(30, 0.5, 5000)` is 5,000 fair coin simulations without using a loop. 
Run the one-call version and watch what survives: the flipping collapses, the 
**counting does not**. And **predict before you run** — the seed is still 41,
so does `n_extreme` come back 50?

In [ ]:
import numpy as np

rng = np.random.default_rng(41)
draws = rng.binomial(30, 0.5, 5000)   # 5,000 whole runs, one call, no flip loop

n_extreme = 0
for heads in draws:                   # the counter does not go away
    if heads >= 22:
        n_extreme += 1
print("n_extreme =", n_extreme, "  p =", round((n_extreme + 1) / 5001, 4))

48, not 50. p = 0.0098 rather than 0.0102. Neither number is the more correct one. 
`count_heads` asks the generator for 30 values per run; `rng.binomial` consumes the 
stream a different amount and in a different way, so seed 41 hands the two
versions different numbers. **A seed reproduces a computation, not an answer** — change how you
draw and you change what you get, which is why every check in this notebook names its exact
call sequence.

Either form is fine to write. The rest of this lab stays with the loop, because the loop is the
version you can read and the question is still the same cycle of loop, `if`, counter, and division.

**Test your understanding.** Write a function named `p_value` that takes four parameters: `observed`, the head count you actually saw; `n_flips`, the number of flips in each run; `n_runs`, the number of fair-coin runs to simulate; and `rng`, a generator. It should simulate `n_runs` fair runs of `n_flips` flips using `count_heads`, count how many reach `observed` heads or more, and return `(n_extreme + 1) / (n_runs + 1)`, the pseudocounted fraction from section 2, rounded to 4 decimals. This is question 2. Its hint is at the end of the notebook.

In [ ]:
# your turn: a function named p_value(observed, n_flips, n_runs, rng)
import numpy as np

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("p_value", expect=1.0, args=(0, 30, 200, np.random.default_rng(7)),
      hint="every run has 0 or more heads — the fraction is forced")
check("p_value", expect=0.0102, args=(22, 30, 5000, np.random.default_rng(41)),
      hint="this should reproduce sections 1-2 exactly: same seed, same draws, "
           "then round to 4 decimals — or return (n+1)/(runs+1) and compare")

## 3 · Twenty coins at once

Simulate twenty fair coins, 30 flips each, and count how many come back with a p-value under 0.05 — even though every single one is actually fair. Predict that number before you run it. (Zero is a common — and wrong — guess.)

In [ ]:
import numpy as np

def count_heads(n_flips, chance, rng):
    heads = 0
    for _ in range(n_flips):
        if rng.random() < chance:
            heads += 1
    return heads

def p_value(observed, n_flips, n_runs, rng):
    n_extreme = 0
    for _ in range(n_runs):
        if count_heads(n_flips, 0.5, rng) >= observed:
            n_extreme += 1
    return (n_extreme + 1) / (n_runs + 1)

rng = np.random.default_rng(43)
flagged = 0
for coin in range(20):
    observed = count_heads(30, 0.5, rng) 
    p = p_value(observed, 30, 500, rng)
    if p < 0.05:
        flagged += 1
print("fair coins flagged:", flagged, "of 20")

One fair coin flagged (your run may differ — rerun with other seeds and watch it wobble around one). Nothing malfunctioned: a 0.05 threshold flags about 5% of fair coins by design, and twenty tests give it twenty chances to do so. This is chapter 2's second half: when you run many tests, some will cross the threshold by chance alone. Unit 9 covers how to correct for that at genome scale.

**Test your understanding.** Write a function named `expected_flags` that takes two parameters: `n_tests`, how many fair tests are run, and `threshold`, the p-value cutoff below which a test is flagged. It should return the number of false flags the threshold produces on average across those fair tests. No randomness is needed. This is question 3. Its hint is at the end of the notebook.

In [ ]:
# your turn: a function named expected_flags(n_tests, threshold)

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("expected_flags", expect=1.0, args=(20, 0.05))
check("expected_flags", expect=400.0, args=(8000, 0.05),
      hint="the number the textbook calls the E[FP] bill")

## If you finish early

- Change section 3's threshold to 0.01 and predict the flag count before rerunning.
- Chapter 2's practice problems run this same machinery on the real scan.

## If you are stuck

- **`n_extreme` misses the expected 50** — the check needs the exact seed-41 stream: create the generator once, then 5,000 `count_heads` calls and nothing else touching it.
- **`p_value` returns something near but not exactly 0.0102** — the check wants 4-decimal rounding; `round(x, 4)`.
- **Section 3 flags far more than one** — check that each coin's `observed` comes from `chance=0.5`; a typo there makes guilty coins.

## Hints

**Question 1 · `n_extreme`.** This is the counter shape from lab 2, built the same way `count_heads` above is built. Set `n_extreme = 0`, then write a `for _ in range(5000):` loop. Inside it, call `count_heads(30, 0.5, rng)` once, and use an `if` to add 1 to `n_extreme` when the result is 22 or more.

**Question 2 · `p_value`.** Question 1's loop is the function body with the fixed numbers replaced by the parameters. Set `n_extreme = 0`, write a `for _ in range(n_runs):` loop, call `count_heads(n_flips, 0.5, rng)` once per pass, and add 1 to `n_extreme` when the result is at least `observed`. After the loop, return `round((n_extreme + 1) / (n_runs + 1), 4)`. Section 3's cell contains a finished version of this function; compare against it after your own attempt.

**Question 3 · `expected_flags`.** One line in the body: return `n_tests * threshold`. Each fair test crosses the threshold with probability equal to the threshold, so section 3's twenty coins at 0.05 flag about 1 on average, and 20 times 0.05 is 1. Chapter 2 calls this number the expected false positives.